# 4 · Cost of the questions notebook 3 just confirmed are answered correctly

Notebook 3 proved these questions get correct, grounded answers. This measures what
that correctness costs — the agent's critique-and-retry loop against a plain
retrieve-then-answer baseline, on the exact same real questions, using the same
persisted index (requires notebook 1, same as notebook 3).

In [ ]:
%%capture
!pip install -q -r requirements.txt


In [ ]:
# Get the project files (config.py, portfolio.py, data/) if they aren't
# already here -- lets this notebook be opened and run on its own in Colab.
import os, subprocess, sys

if not os.path.exists("portfolio.py"):
    if os.path.exists("../portfolio.py"):
        os.chdir("..")
    else:
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/hossamhamdy333/AI_Portfolio.git", "repo"],
            check=True,
        )
        os.chdir("repo/Codebase_Insight_Agent")

sys.path.insert(0, os.getcwd())
print("Working directory:", os.getcwd())


In [ ]:
from getpass import getpass

if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Google API key (Gemini): ")

# QDRANT_URL/QDRANT_API_KEY make the index PERSISTENT (Qdrant Cloud, free
# tier is enough) instead of rebuilt from scratch every time. This matters
# specifically because this notebook runs in a fresh Colab VM each time,
# separate from wherever mcp_server.py/web_app.py actually run - without a
# real, shared QDRANT_URL, this notebook's work never reaches those
# services at all. Get a free instance at https://cloud.qdrant.io
if not os.environ.get("QDRANT_URL"):
    os.environ["QDRANT_URL"] = input("Qdrant Cloud URL (blank = local in-memory, no persistence): ")
if os.environ["QDRANT_URL"] and not os.environ.get("QDRANT_API_KEY"):
    os.environ["QDRANT_API_KEY"] = getpass("Qdrant API key: ")


In [ ]:
import config
import portfolio

indexes = portfolio.load_all_indexes()  # requires notebook 1 to have been run
router = portfolio.build_router()
agent = portfolio.build_agent(indexes, router)


In [ ]:
import pandas as pd

eval_questions = pd.read_csv("data/eval_set_starter.csv")
print(f"{len(eval_questions)} regression questions loaded")
eval_questions[["question", "expected_projects"]]


In [ ]:
rows = []

for i, row in eval_questions.iterrows():
    question = row["question"]
    # naive_ask takes a single index; for the handful of rows that expect
    # two projects at once, the first one is the baseline target - a real
    # simplification, not a bug: the point of this comparison is cost, and
    # the agent still gets evaluated against its full multi-project answer,
    # only the single-index baseline is narrowed.
    target = row["expected_projects"].split(",")[0].strip()

    agent_result = portfolio.ask(agent, question, thread_id=f"eval-{i}")

    baseline_index = indexes[target] if target in indexes else indexes[agent_result["projects"][0]]
    baseline_result = portfolio.naive_ask(baseline_index, question)

    rows.append({
        "question": question[:55] + ("..." if len(question) > 55 else ""),
        "baseline_llm_calls": baseline_result["llm_calls"],
        "agent_llm_calls": agent_result["llm_calls"],
    })

comparison = pd.DataFrame(rows)
comparison


In [ ]:
import matplotlib.pyplot as plt

x = range(len(comparison))
plt.figure(figsize=(8, 4.5))
plt.bar([i - 0.2 for i in x], comparison["baseline_llm_calls"], width=0.4, label="baseline", color="#C9CDD3")
plt.bar([i + 0.2 for i in x], comparison["agent_llm_calls"], width=0.4, label="agent", color="#4C8BF5")
plt.xticks(list(x), [f"Q{i+1}" for i in x])
plt.ylabel("LLM calls")
plt.title("Cost of the critique loop, per question - on questions already confirmed correct")
plt.legend()
plt.tight_layout()
plt.show()


**Worth recording this number somewhere** (a comment in this cell, a note in your
own tracking) each time you run this after a `force=True` reindex — a sudden jump in
average `agent_llm_calls` without a corresponding change to your data usually means the
critique loop is failing more often, which is worth investigating even if notebook 3's
pass/fail results still look fine.